In [1]:
%pip install duckdb


   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.3 MB 11.3 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/12.3 MB 16.7 MB/s eta 0:00:01
   --- ------------------------------------ 1.0/12.3 MB 16.7 MB/s eta 0:00:01
   --- ------------------------------------ 1.0/12.3 MB 16.7 MB/s eta 0:00:01
   --- ------------------------------------ 1.1/12.3 MB 4.6 MB/s eta 0:00:03
   ------ --------------------------------- 1.9/12.3 MB 6.6 MB/s eta 0:00:02
   ------ --------------------------------- 2.1/12.3 MB 7.0 MB/s eta 0:00:02
   ------ --------------------------------- 2.1/12.3 MB 7.0 MB/s eta 0:00:02
   ------ --------------------------------- 2.1/12.3 MB 7.0 MB/s eta 0:00:02
   --------- ------------------------------ 2.8/12.3 MB 6.1 MB/s eta 0:00:02
   ---------- ----------------------------- 3.1/12.3 MB 6.5 MB/s eta 0:00:02
   ---------- ----------------------------- 3.1/12.3 MB 6.5 MB/s eta 0:00:02
 

In [2]:
from pathlib import Path
import duckdb

project_folder = Path(r"D:\electricity-tariff-optimisation")
raw_folder = project_folder / "data" / "raw"
processed_folder = project_folder / "data" / "processed"

csv_files = sorted(raw_folder.rglob("*.csv"))

if len(csv_files) != 168:
    raise ValueError(
        f"Expected 168 source CSV files, found {len(csv_files)}."
    )

connection = duckdb.connect(
    str(processed_folder / "electricity_analysis.duckdb")
)

connection.execute("SET memory_limit = '1GB'")
connection.execute("SET threads = 2")

print("DuckDB version:", duckdb.__version__)
print("Source CSV files:", len(csv_files))
print("Database connection ready.")

DuckDB version: 1.4.5
Source CSV files: 168
Database connection ready.


In [3]:
connection.read_csv(
    [str(file) for file in csv_files],
    header=True,
    all_varchar=True,
).create_view("raw_consumption")

print("Source columns:")
display(connection.sql("DESCRIBE raw_consumption").df())

customer_inventory = connection.execute("""
    WITH parsed AS (
        SELECT
            TRIM(LCLid) AS household_id,
            TRIM(stdorToU) AS tariff_group,
            TRY_CAST(DateTime AS TIMESTAMP) AS interval_end
        FROM raw_consumption
    )
    SELECT
        household_id,
        COUNT(*) AS raw_rows_2013,
        MIN(interval_end) AS first_interval_end,
        MAX(interval_end) AS last_interval_end
    FROM parsed
    WHERE tariff_group = 'Std'
      AND interval_end > TIMESTAMP '2013-01-01 00:00:00'
      AND interval_end <= TIMESTAMP '2014-01-01 00:00:00'
    GROUP BY household_id
    ORDER BY household_id
""").df()

inventory_file = (
    processed_folder / "standard_customer_inventory_2013.csv"
)

customer_inventory.to_csv(inventory_file, index=False)

print("Standard-tariff households found:", len(customer_inventory))
print("Inventory saved:", inventory_file)

display(customer_inventory.head(10))

Source columns:


,column_name,column_type,null,key,default,extra
0,LCLid,VARCHAR,YES,None,None,None
1,stdorToU,VARCHAR,YES,None,None,None
2,DateTime,VARCHAR,YES,None,None,None
3,KWH/hh (per half hour),VARCHAR,YES,None,None,None


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Standard-tariff households found: 4411
Inventory saved: D:\electricity-tariff-optimisation\data\processed\standard_customer_inventory_2013.csv


,household_id,raw_rows_2013,first_interval_end,last_interval_end
0,MAC000002,17532,2013-01-01 00:30:00,2014-01-01
1,MAC000003,17530,2013-01-01 00:30:00,2014-01-01
2,MAC000004,17531,2013-01-01 00:30:00,2014-01-01
3,MAC000006,17532,2013-01-01 00:30:00,2014-01-01
4,MAC000007,17529,2013-01-01 00:30:00,2014-01-01
5,MAC000008,14504,2013-01-01 00:30:00,2013-10-30
6,MAC000009,17523,2013-01-01 00:30:00,2014-01-01
7,MAC000010,17532,2013-01-01 00:30:00,2014-01-01
8,MAC000011,17531,2013-01-01 00:30:00,2014-01-01
9,MAC000012,17531,2013-01-01 00:30:00,2014-01-01


In [4]:
import pandas as pd

customer_inventory["first_interval_end"] = pd.to_datetime(
    customer_inventory["first_interval_end"]
)

customer_inventory["last_interval_end"] = pd.to_datetime(
    customer_inventory["last_interval_end"]
)

candidate_pool = customer_inventory.loc[
    customer_inventory["first_interval_end"].eq(
        pd.Timestamp("2013-01-01 00:30:00")
    )
    & customer_inventory["last_interval_end"].eq(
        pd.Timestamp("2014-01-01 00:00:00")
    )
    & customer_inventory["raw_rows_2013"].ge(17520)
].copy()

candidate_customers = (
    candidate_pool.sort_values("household_id")
    .sample(
        n=min(1000, len(candidate_pool)),
        random_state=42,
    )
    .sort_values("household_id")
    .reset_index(drop=True)
)

candidate_file = (
    processed_folder / "candidate_standard_customers_2013.csv"
)

candidate_customers.to_csv(candidate_file, index=False)

connection.register(
    "candidate_households",
    candidate_customers[["household_id"]],
)

print("Households in preliminary pool:", len(candidate_pool))
print("Candidate households sampled:", len(candidate_customers))
print("Random seed: 42")
print("Candidate list saved:", candidate_file)

display(candidate_customers.head(10))

Households in preliminary pool: 3594
Candidate households sampled: 1000
Random seed: 42
Candidate list saved: D:\electricity-tariff-optimisation\data\processed\candidate_standard_customers_2013.csv


,household_id,raw_rows_2013,first_interval_end,last_interval_end
0,MAC000002,17532,2013-01-01 00:30:00,2014-01-01
1,MAC000011,17531,2013-01-01 00:30:00,2014-01-01
2,MAC000020,17530,2013-01-01 00:30:00,2014-01-01
3,MAC000022,17531,2013-01-01 00:30:00,2014-01-01
4,MAC000025,17526,2013-01-01 00:30:00,2014-01-01
5,MAC000033,17531,2013-01-01 00:30:00,2014-01-01
6,MAC000036,17531,2013-01-01 00:30:00,2014-01-01
7,MAC000037,17531,2013-01-01 00:30:00,2014-01-01
8,MAC000040,17532,2013-01-01 00:30:00,2014-01-01
9,MAC000041,17530,2013-01-01 00:30:00,2014-01-01


In [5]:
source_columns = connection.sql(
    "DESCRIBE raw_consumption"
).df()["column_name"].tolist()

energy_matches = [
    name
    for name in source_columns
    if name.strip() == "KWH/hh (per half hour)"
]

if len(energy_matches) != 1:
    raise ValueError("Could not identify a unique energy column.")

energy_identifier = (
    '"' + energy_matches[0].replace('"', '""') + '"'
)

connection.execute(f"""
    CREATE OR REPLACE TABLE candidate_consumption_2013 AS
    WITH parsed AS (
        SELECT
            TRIM(r.LCLid) AS household_id,
            TRIM(r.stdorToU) AS tariff_group,
            TRY_CAST(r.DateTime AS TIMESTAMP) AS interval_end_utc,
            TRIM(r.{energy_identifier}) AS raw_energy_value
        FROM raw_consumption AS r
        INNER JOIN candidate_households AS c
            ON TRIM(r.LCLid) = c.household_id
        WHERE TRIM(r.stdorToU) = 'Std'
    )
    SELECT
        *,
        TRY_CAST(raw_energy_value AS DOUBLE) AS energy_kwh
    FROM parsed
    WHERE interval_end_utc > TIMESTAMP '2013-01-01 00:00:00'
      AND interval_end_utc <= TIMESTAMP '2014-01-01 00:00:00'
""")

connection.execute("CHECKPOINT")

stored_summary = connection.sql("""
    SELECT
        COUNT(*) AS raw_rows,
        COUNT(DISTINCT household_id) AS households,
        MIN(interval_end_utc) AS first_interval_end,
        MAX(interval_end_utc) AS last_interval_end,
        COUNT(*) FILTER (
            WHERE energy_kwh IS NULL
        ) AS missing_or_invalid_readings
    FROM candidate_consumption_2013
""").df()

display(stored_summary)

print("Candidate records stored in the local DuckDB database.")
print("Duplicates and invalid readings have not been removed.")
print("Timestamp values represent GMT/UTC interval ends.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,raw_rows,households,first_interval_end,last_interval_end,missing_or_invalid_readings
0,17529891,1000,2013-01-01 00:30:00,2014-01-01,0


Candidate records stored in the local DuckDB database.
Duplicates and invalid readings have not been removed.
Timestamp values represent GMT/UTC interval ends.


In [6]:
connection.execute("""
    CREATE OR REPLACE TABLE candidate_slot_quality_2013 AS
    SELECT
        household_id,
        interval_end_utc,
        COUNT(*) AS source_rows,
        COUNT(DISTINCT energy_kwh) AS distinct_readings,
        COUNT(*) FILTER (
            WHERE energy_kwh IS NULL
               OR NOT isfinite(energy_kwh)
               OR energy_kwh < 0
        ) AS invalid_readings,
        MIN(energy_kwh) AS energy_kwh
    FROM candidate_consumption_2013
    GROUP BY household_id, interval_end_utc
""")

customer_quality = connection.sql("""
    SELECT
        household_id,
        SUM(source_rows) AS raw_rows,
        COUNT(*) AS unique_slots,
        SUM(source_rows - 1) AS duplicate_extra_rows,
        COUNT(*) FILTER (
            WHERE distinct_readings > 1
        ) AS conflicting_slots,
        SUM(invalid_readings) AS invalid_readings,
        COUNT(*) FILTER (
            WHERE EXTRACT(SECOND FROM interval_end_utc) <> 0
               OR EXTRACT(MINUTE FROM interval_end_utc) NOT IN (0, 30)
        ) AS off_grid_slots,
        COUNT(*) FILTER (
            WHERE energy_kwh = 0
        ) AS zero_slots
    FROM candidate_slot_quality_2013
    GROUP BY household_id
    ORDER BY household_id
""").df()

customer_quality["complete_valid"] = (
    customer_quality["unique_slots"].eq(17520)
    & customer_quality["conflicting_slots"].eq(0)
    & customer_quality["invalid_readings"].eq(0)
    & customer_quality["off_grid_slots"].eq(0)
)

customer_quality["zero_pct"] = (
    customer_quality["zero_slots"]
    / customer_quality["unique_slots"] * 100
)

quality_folder = project_folder / "reports" / "data_quality"
quality_folder.mkdir(parents=True, exist_ok=True)

customer_quality.to_csv(
    quality_folder / "candidate_customer_quality_2013.csv",
    index=False,
)

print("Households checked:", len(customer_quality))
print(
    "Complete households with valid, non-conflicting readings:",
    int(customer_quality["complete_valid"].sum()),
)
print(
    "Households with conflicting readings:",
    int(customer_quality["conflicting_slots"].gt(0).sum()),
)

display(customer_quality.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Households checked: 1000
Complete households with valid, non-conflicting readings: 248
Households with conflicting readings: 0


,household_id,raw_rows,unique_slots,duplicate_extra_rows,conflicting_slots,invalid_readings,off_grid_slots,zero_slots,complete_valid,zero_pct
0,MAC000002,17532.0,17520,12.0,0,0.0,0,0,True,0.000000
1,MAC000011,17531.0,17519,12.0,0,0.0,0,0,False,0.000000
2,MAC000020,17530.0,17518,12.0,0,0.0,0,0,False,0.000000
3,MAC000022,17531.0,17519,12.0,0,0.0,0,0,False,0.000000
4,MAC000025,17526.0,17514,12.0,0,0.0,0,48,False,0.274066
5,MAC000033,17531.0,17519,12.0,0,0.0,0,0,False,0.000000
6,MAC000036,17531.0,17519,12.0,0,0.0,0,2198,False,12.546378
7,MAC000037,17531.0,17519,12.0,0,0.0,0,17417,False,99.417775
8,MAC000040,17532.0,17520,12.0,0,0.0,0,0,True,0.000000
9,MAC000041,17530.0,17518,12.0,0,0.0,0,873,False,4.983446


In [7]:
selected_customers = (
    customer_quality.loc[
        customer_quality["complete_valid"],
        ["household_id", "zero_pct"],
    ]
    .sort_values("household_id")
    .reset_index(drop=True)
)

connection.register(
    "selected_household_ids",
    selected_customers[["household_id"]],
)

connection.execute("""
    CREATE OR REPLACE TABLE analysis_consumption_2013 AS
    SELECT
        q.household_id,
        q.interval_end_utc,
        q.interval_end_utc - INTERVAL '30 minutes'
            AS interval_start_utc,
        q.energy_kwh
    FROM candidate_slot_quality_2013 AS q
    INNER JOIN selected_household_ids AS s
        ON q.household_id = s.household_id
    WHERE q.distinct_readings = 1
      AND q.invalid_readings = 0
""")

analysis_summary = connection.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT household_id) AS households,
        SUM(energy_kwh) AS total_energy_kwh
    FROM analysis_consumption_2013
""").df()

expected_rows = len(selected_customers) * 17520

assert int(analysis_summary.loc[0, "rows"]) == expected_rows
assert int(analysis_summary.loc[0, "households"]) == len(selected_customers)

selected_customers.to_csv(
    processed_folder / "selected_complete_customers_2013.csv",
    index=False,
)

connection.execute("CHECKPOINT")

display(analysis_summary.round(3))

print("Selected households:", len(selected_customers))
print("Expected rows:", expected_rows)
print("UTC interval timestamps retained.")
print("Zero readings retained; no missing values imputed.")

print("\nZero-reading percentage among selected households:")
display(
    selected_customers["zero_pct"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ).round(2)
)

,rows,households,total_energy_kwh
0,4344960,248,1038488.042


Selected households: 248
Expected rows: 4344960
UTC interval timestamps retained.
Zero readings retained; no missing values imputed.

Zero-reading percentage among selected households:


count    248.00
mean       0.64
std        4.55
min        0.00
50%        0.00
90%        0.03
95%        1.81
99%       12.42
max       65.62
Name: zero_pct, dtype: float64

In [8]:
highest_zero = (
    selected_customers
    .sort_values(
        ["zero_pct", "household_id"],
        ascending=[False, True],
    )
    .head(10)
)

connection.register(
    "high_zero_households",
    highest_zero[["household_id"]],
)

monthly_zero_review = connection.sql("""
    SELECT
        a.household_id,
        DATE_TRUNC('month', a.interval_start_utc) AS month_utc,
        COUNT(*) AS intervals,
        SUM(a.energy_kwh) AS total_kwh,
        100.0 * COUNT(*) FILTER (
            WHERE a.energy_kwh = 0
        ) / COUNT(*) AS zero_pct
    FROM analysis_consumption_2013 AS a
    INNER JOIN high_zero_households AS h
        ON a.household_id = h.household_id
    GROUP BY a.household_id, month_utc
    ORDER BY a.household_id, month_utc
""").df()

print("Households with the highest zero-reading percentages:")
display(highest_zero.round(2))

print("\nMonthly zero-reading percentages:")
display(
    monthly_zero_review.pivot(
        index="household_id",
        columns="month_utc",
        values="zero_pct",
    ).round(1)
)

monthly_zero_review.to_csv(
    quality_folder / "high_zero_households_monthly_review_2013.csv",
    index=False,
)

Households with the highest zero-reading percentages:


,household_id,zero_pct
140,MAC003327,65.62
101,MAC002400,16.52
56,MAC001340,13.17
217,MAC004954,11.59
187,MAC004233,10.82
230,MAC005263,9.85
159,MAC003797,6.34
67,MAC001550,4.68
111,MAC002718,4.36
92,MAC002259,3.88



Monthly zero-reading percentages:


month_utc,2013-01-01,2013-02-01,2013-03-01,2013-04-01,2013-05-01,2013-06-01,2013-07-01,2013-08-01,2013-09-01,2013-10-01,2013-11-01,2013-12-01
household_id,,,,,,,,,,,,
MAC001340,0.0,0.0,0.0,0.0,1.7,0.3,0.0,0.0,0.0,0.1,54.8,100.0
MAC001550,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.6,36.2
MAC002259,6.8,9.4,9.5,2.8,0.5,0.1,0.0,0.1,3.3,3.2,3.4,7.7
MAC002400,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,100.0,89.7,0.0
MAC002718,5.0,3.6,0.0,0.0,0.0,6.9,3.7,7.5,3.8,5.0,8.2,8.5
MAC003327,63.0,57.8,54.9,62.5,73.8,73.6,60.1,66.6,75.0,67.1,66.2,66.5
MAC003797,5.2,1.3,2.5,5.4,3.0,5.7,4.0,27.4,3.8,5.9,6.6,4.8
MAC004233,28.2,41.7,12.0,11.0,5.9,8.0,1.4,2.4,4.9,6.0,4.8,6.1
MAC004954,0.0,0.0,0.0,0.0,0.0,12.7,14.9,21.1,25.8,27.4,20.8,15.8


In [9]:
monthly_quality = connection.sql("""
    SELECT
        household_id,
        DATE_TRUNC('month', interval_start_utc) AS month_utc,
        100.0 * COUNT(*) FILTER (
            WHERE energy_kwh = 0
        ) / COUNT(*) AS zero_pct
    FROM analysis_consumption_2013
    GROUP BY household_id, month_utc
""").df()

max_monthly_zero = (
    monthly_quality.groupby("household_id")["zero_pct"]
    .max()
    .rename("max_monthly_zero_pct")
)

sample_review = selected_customers.merge(
    max_monthly_zero,
    on="household_id",
    how="left",
    validate="one_to_one",
)

sample_review["annual_zero_flag"] = (
    sample_review["zero_pct"].ge(50)
)

sample_review["monthly_zero_flag"] = (
    sample_review["max_monthly_zero_pct"].ge(95)
)

sample_review["review_flag"] = (
    sample_review["annual_zero_flag"]
    | sample_review["monthly_zero_flag"]
)

sample_review.to_csv(
    quality_folder / "selected_customer_review_flags_2013.csv",
    index=False,
)

monthly_quality.to_csv(
    quality_folder / "selected_customer_monthly_zero_2013.csv",
    index=False,
)

print("Main sample households:", len(sample_review))
print("Households flagged for review:", int(sample_review["review_flag"].sum()))
print("Flags indicate unusual patterns, not confirmed data errors.")

display(
    sample_review.loc[sample_review["review_flag"]].round(2)
)

Main sample households: 248
Households flagged for review: 4
Flags indicate unusual patterns, not confirmed data errors.


,household_id,zero_pct,max_monthly_zero_pct,annual_zero_flag,monthly_zero_flag,review_flag
56,MAC001340,13.17,100.0,False,True,True
101,MAC002400,16.52,100.0,False,True,True
140,MAC003327,65.62,75.0,True,False,True
230,MAC005263,9.85,100.0,False,True,True


In [10]:
connection.register("sample_review_input", sample_review)

connection.execute("""
    CREATE OR REPLACE TABLE selected_customer_review_2013 AS
    SELECT *
    FROM sample_review_input
""")

sample_check = connection.sql("""
    SELECT
        COUNT(*) AS households,
        COUNT(*) FILTER (
            WHERE review_flag
        ) AS flagged_households,
        COUNT(*) FILTER (
            WHERE NOT review_flag
        ) AS unflagged_households
    FROM selected_customer_review_2013
""").df()

assert int(sample_check.loc[0, "households"]) == 248
assert int(sample_check.loc[0, "flagged_households"]) == 4
assert int(sample_check.loc[0, "unflagged_households"]) == 244

connection.execute("CHECKPOINT")

display(sample_check)

connection.close()

print("Sample preparation saved.")
print("Database connection closed.")

,households,flagged_households,unflagged_households
0,248,4,244


Sample preparation saved.
Database connection closed.
